# Кластеризация — эксперименты

**Этот ноутбук НЕ перезапускает пайплайн.** Данные грузятся из `artifacts/` за секунды.

Чтобы обновить данные → запусти `pipeline_clustering.ipynb` (ячейку сохранения).

---
| Файл | Что внутри |
|---|---|
| `artifacts/df_for_model_cluster.parquet` | DataFrame, 436k гостей × 14 признаков |
| `artifacts/X_scaled.npy` | numpy array для передачи в KMeans |
| `artifacts/preprocessor.joblib` | ColumnTransformer (fit на обучающих данных) |
| `artifacts/feature_meta.joblib` | Имена признаков, X_NUM / X_CAT / X_BIN |

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# ── Загрузка артефактов ───────────────────────────────────────────────────
df  = pd.read_parquet('artifacts/df_for_model_cluster.parquet')
X_scaled     = np.load('artifacts/X_scaled.npy')
preprocessor = joblib.load('artifacts/preprocessor.joblib')
meta         = joblib.load('artifacts/feature_meta.joblib')

feature_names_out = meta['feature_names_out']
X_NUM = meta['X_NUM']
X_CAT = meta['X_CAT']
X_BIN = meta['X_BIN']

print(f'df:       {df.shape[0]:,} гостей × {df.shape[1]} признаков')
print(f'X_scaled: {X_scaled.shape[0]:,} × {X_scaled.shape[1]}')
print(f'Признаки: {feature_names_out}')

## Обучение модели

In [ ]:
# ── Параметры эксперимента — меняй здесь ─────────────────────────────────
K = 5              # число кластеров
RANDOM_STATE = 42
# ─────────────────────────────────────────────────────────────────────────

km = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=RANDOM_STATE)
km.fit(X_scaled)

df['cluster'] = km.labels_

print(f'Inertia:    {km.inertia_:.0f}')
print(f'Silhouette: {silhouette_score(X_scaled, km.labels_, sample_size=50_000, random_state=42):.4f}')
print(f'CH Score:   {calinski_harabasz_score(X_scaled, km.labels_):.1f}')
print(f'DB Score:   {davies_bouldin_score(X_scaled, km.labels_):.4f}')

df['cluster'].value_counts().sort_index()

## Профили кластеров

In [ ]:
# Средние значения числовых признаков по кластерам
profile_num = df.groupby('cluster')[X_NUM].mean().round(2)
profile_num

In [ ]:
# Распределение категориальных признаков по кластерам
for col in X_CAT:
    print(f'\n=== {col} ===')
    print(
        df.groupby('cluster')[col]
          .value_counts(normalize=True)
          .mul(100).round(1)
          .rename('% гостей')
    )

In [ ]:
# Тепловая карта числовых профилей
fig, ax = plt.subplots(figsize=(12, max(3, K)))
sns.heatmap(
    profile_num.T,
    annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.5, ax=ax
)
ax.set_title(f'Профили кластеров (K={K})', fontsize=13)
plt.tight_layout()
plt.show()

## Сохранение модели (опционально)

In [ ]:
# Раскомментируй, чтобы сохранить выбранную модель
# joblib.dump(km, f'artifacts/kmeans_k{K}.joblib')
# df.to_parquet(f'artifacts/df_with_clusters_k{K}.parquet')
# print('Модель и датасет с метками кластеров сохранены')